# 🗓 Calendar Agent — Lightning AI Studio LoRA Training

Lightning AI Studio에서 `calendar-agent` 모델을 LoRA 학습·평가하는 노트북입니다. 데이터·코드는 Git 저장소에 포함되어 `git pull`로 따라옵니다 (별도 데이터 업로드 불필요).

## 💡 실행 전 준비
1. **GPU 확인**: 우측 상단에서 GPU(L4 / A10G / T4)가 켜져 있는지 확인하세요. (학습하지 않을 때는 CPU로 전환해 크레딧 절약)
2. **최신 코드 받기**: 터미널에서 `git pull origin main`으로 최신 라운드를 받은 뒤, 셀을 위→아래로 실행하세요.

## 0. GPU 작동 확인
A10G 또는 Tesla T4 등이 정상적으로 인식되는지 확인합니다.

In [ ]:
import subprocess
subprocess.run(["nvidia-smi"])

## 1. 경로 설정 및 이동
Lightning AI Studio의 기본 작업 디렉토리는 `/teamspace/studios/this_studio` 입니다. 프로젝트 폴더로 이동합니다.

In [ ]:
import os

# 프로젝트 폴더로 이동 (git clone/pull로 받은 경로)
default_path = '/teamspace/studios/this_studio'
project_name = 'calendar-agent'
target = os.path.join(default_path, project_name)

if os.path.isdir(target):
    os.chdir(target)
elif os.path.basename(os.getcwd()) != project_name:
    print(f"[주의] {project_name} 폴더를 찾을 수 없습니다. 현재 경로: {os.getcwd()}")

print("작업 경로:", os.getcwd())

## 2. 의존성 설치 및 캐시 클린업
프로젝트를 개발용 패키지로 설치하고, 학습에 불필요한 패키지를 정리합니다.

In [ ]:
# 2. 의존성 설치 — 이미 설치돼 있으면 건너뜀 (Lightning은 패키지가 커널/세션 재시작에도 영구 보존)
# %pip 매직 대신 sys.executable -m pip 사용: 같은 커널 환경에 설치 + 순수 Python(린터 안전).
import importlib.util, os, sys, subprocess

if importlib.util.find_spec("trl") is None:
    print("학습 의존성 설치 중... (최초 1회)")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[train]"], check=True)
else:
    print("의존성 이미 설치됨 — 설치 건너뜀")

os.environ["WANDB_DISABLED"] = "true"
import torch
print("torch:", torch.__version__, "cuda:", torch.cuda.is_available(), "ngpu:", torch.cuda.device_count())

## 3. 모델 학습 실행
먼저 **라운드 확인 셀**을 실행하면 현재 라운드·데이터가 학습 없이 바로 표시됩니다. 확인 후 **학습 실행 셀**을 돌리세요. 라운드는 확인 셀의 `CONFIG` 한 줄로 바꿉니다.

* **Qwen3-0.6B (r29 A/B, 기본)**: `configs/train_lightning_qwen3_0_6b.yaml`
* **0.5B (fallback)**: `configs/train.yaml`

In [ ]:
# ── 라운드 확인 (이 셀만 실행하면 학습 없이 현재 라운드가 바로 표시됨) ──
CONFIG = 'configs/train_lightning_qwen3_0_6b.yaml'  # 학습할 라운드. 0.5B(fallback): configs/train.yaml

import yaml
_cfg  = yaml.safe_load(open(CONFIG, encoding='utf-8'))
_mcfg = yaml.safe_load(open(_cfg['model_config'], encoding='utf-8'))

# yaml을 읽은 직후 바로 라운드 정보 표시
print(f"★ 현재 라운드 : {_cfg['run_name']}")
print(f"  베이스 모델 : {_mcfg['hf_id']}")
print(f"  output_dir : {_cfg['output_dir']}")
print(f"  유효 배치   : {_cfg['per_device_train_batch_size'] * _cfg['gradient_accumulation_steps']}")
print(f"  epochs / lr : {_cfg['num_train_epochs']} / {_cfg['learning_rate']}")

In [ ]:
# ── 학습 실행 (위 '라운드 확인' 셀을 먼저 실행해 CONFIG를 설정한 뒤 돌리세요) ──
import sys, subprocess, torch

print(f"학습 시작 → {CONFIG}  (라운드: {_cfg['run_name']})")
num_gpus = torch.cuda.device_count()
if num_gpus > 1:
    print(f"멀티 GPU {num_gpus}개 감지: torchrun으로 분산 학습합니다.")
    cmd = ["torchrun", f"--nproc_per_node={num_gpus}", "scripts/train_lora.py", "--config", CONFIG]
else:
    print("싱글 GPU 감지: 단일 프로세스로 학습합니다.")
    cmd = [sys.executable, "scripts/train_lora.py", "--config", CONFIG]
subprocess.run(cmd, check=True)

## 4. 모델 병합 및 평가
학습된 LoRA 어댑터를 베이스 모델과 병합(Merge)한 뒤, 골든셋으로 성능을 평가합니다. (Qwen3는 `eval_model.py`가 thinking 토큰을 자동 비활성화해 순수 JSON으로 채점합니다.)

In [ ]:
import os, sys, subprocess, yaml

# 설정 파싱 — CONFIG에서 base/경로 자동 인식
_cfg  = yaml.safe_load(open(CONFIG, encoding='utf-8'))
_mcfg = yaml.safe_load(open(_cfg['model_config'], encoding='utf-8'))
BASE       = _mcfg['hf_id']
LORA_DIR   = _cfg['output_dir']
NAME       = os.path.basename(LORA_DIR)
MERGED_DIR = f'models/merged/{NAME}'
EVAL_JSON  = f'logs/eval_{NAME}.json'
GOLDEN     = _cfg.get('eval_golden', 'data/eval/golden.jsonl')
print("베이스 모델:", BASE)
print("LoRA 경로 :", LORA_DIR)
print("병합 대상 :", MERGED_DIR)
print("골든셋    :", GOLDEN)

# 1. LoRA 병합 → 2. 골든셋 평가 (eval_model.py가 Qwen3 thinking 자동 비활성화)
subprocess.run([sys.executable, "scripts/merge_lora.py", "--base", BASE, "--lora", LORA_DIR, "--out", MERGED_DIR], check=True)
subprocess.run([sys.executable, "scripts/eval_model.py", "--model", MERGED_DIR, "--eval", GOLDEN, "--out", EVAL_JSON], check=True)

print(f"--- 평가 결과 ({EVAL_JSON}) ---")
print(open(EVAL_JSON, encoding='utf-8').read() if os.path.exists(EVAL_JSON) else "평가 결과 파일을 찾을 수 없습니다.")

## 5. 학습 결과 백업 안내

> [!NOTE]
> 학습된 LoRA 가중치는 Lightning Studio 내부 스토리지(위 `output_dir`, 예: `models/lora/r29-qwen3-0.6b-lightning`)에 보존됩니다.
> 로컬 PC로 가져오려면 왼쪽 파일 탐색기에서 해당 폴더를 우클릭 → **[Download]** 하세요.
> (이후 로컬에서 `merge_lora.py` → `quantize.sh`로 gguf 변환.)